In [44]:
from rag_helper import RAGBase
from ingest import load_course_lessons, build_index

In [45]:
documents = load_course_lessons()

# Question 1

In [46]:
len(documents)

72

In [47]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

# Question 2

In [48]:
index = build_index(documents)

In [49]:
question = "How does the agentic loop keep calling the model until it stops?"

In [50]:
search_results = index.search(
    question,
    boost_dict={'content': 3.0},
    num_results=5
)

search_results[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

In [51]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

# Question 3

In [52]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [53]:
USER_PROMPT_TEMPALATE = '''
Question:
{question}

Context:
{context}
'''

In [54]:
question = 'How does the agentic loop keep calling the model until it stops?'

In [55]:
search_results = index.search(
    question,
    boost_dict={'content': 3.0},
    num_results=5
)

In [56]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append('Filename: ' + doc['filename'])
        lines.append('Content: ' + doc['content'])

    return '\n'.join(lines).strip()

In [57]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPALATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [58]:
prompt = build_prompt(question, search_results)

In [59]:
prompt

'Question:\nHow does the agentic loop keep calling the model until it stops?\n\nContext:\nFilename: 01-agentic-rag/lessons/14-agentic-loop.md\nContent: # The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as t

In [60]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=prompt
)

In [61]:
response.output_text

'It keeps calling the model with a `while True` loop and stops only when the latest model response contains **no function calls**.\n\nIn the lesson’s code:\n\n1. Call the model with the current `messages`.\n2. Check each item in `response.output`.\n3. If there’s a `function_call`, run the tool, append the tool result to `messages`, and set `has_function_calls = True`.\n4. If there are **no** function calls in that turn, `has_function_calls` stays `False`.\n5. Then the loop does:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nSo the stopping rule is simple:\n\n- **function call present** → run tool, loop again\n- **no function calls** → final answer, stop\n\nThat’s how the model keeps getting called until it’s done.'

In [62]:
response.usage

ResponseUsage(input_tokens=7083, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=179, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7262)

In [63]:
assistant = RAGBase(index, openai_client)

In [64]:
response = assistant.rag(question)

In [65]:
response['answer']

'It keeps calling the model inside a `while True` loop and checks whether the model returned any `function_call` items.\n\n- If there is a function call, the code runs the tool, appends the tool result to `messages`, and loops again.\n- If there are no function calls in the response, it `break`s out of the loop.\n\nSo the stop condition is: **no function calls this turn**.'

In [66]:
response['usage']

ResponseUsage(input_tokens=7136, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=90, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7226)

# Question 4

In [67]:
from gitsource import chunk_documents

In [68]:
chunks = chunk_documents(documents, size=2000, step=1000)

In [69]:
len(chunks)

295

# Question 5

In [70]:
index = build_index(chunks)

In [71]:
search_results = index.search(
    question,
    boost_dict={'content': 3.0},
    num_results=5
)

In [72]:
assistant = RAGBase(index, openai_client)

In [73]:
response = assistant.rag(question)

In [74]:
response['answer']

"It keeps calling the model inside a `while True` loop, and after each turn it checks whether the model returned any `function_call` items.\n\n- If there are function calls, the code runs them, appends the tool results to `messages`, and loops again.\n- If there are no function calls in that turn, it breaks out of the loop and stops.\n\nSo the stop condition is: **no function calls this turn means we're done**."

In [75]:
response['usage']

ResponseUsage(input_tokens=2319, input_tokens_details=InputTokensDetails(cached_tokens=1792), output_tokens=96, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2415)

# Question 6

In [76]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [82]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the chunked document for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course lessons.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [83]:
def search(query: str) -> dict[str, str]:
    """
    Search the chunked document for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'content': 3.0}
    )

In [84]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [85]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [87]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

In [88]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [89]:
result = runner.loop(
    prompt=question,
    callback=callback,
)

-> Response received


-> Response received


In [90]:
result.cost

CostInfo(input_cost=Decimal('0.006156'), output_cost=Decimal('0.001062'), total_cost=Decimal('0.007218'))

In [91]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How does the agentic loop keep calling the model until it stops?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop keeps c

In [ ]:
runner.run()

You: how does agent loop work?


-> Response received


-> Response received
